# E4 港式票据 · 篡改的定性演示

工行杯 · 金融安全服务方向

E3 在 CORD 上给统计量，E4 在港式票据上给**画面**。两件 E3 做不到的事：

1. **校验三**（折扣标签自带金额）只在港式票据上成立——CORD 的印尼票据没有这种冗余
2. **字形复制式篡改**视觉上几乎看不出来，比整块重绘更能说明"肉眼靠不住"

票据从公开仓库下载，篡改样本在本 notebook 里现场生成，不需要上传任何文件。


## 1. 装依赖、配 API、取数据

In [ ]:
!pip install -q langchain-core langchain-deepseek

import os
import urllib.request
from pathlib import Path

try:
    from google.colab import userdata
    os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
except Exception as e:
    raise SystemExit(f"请先在左栏钥匙图标里加 DEEPSEEK_API_KEY：{e}")

REPO = "https://raw.githubusercontent.com/baimingyang98/FTEC5660/main/public_test"
WORK = Path("/content/ghb")
(WORK / "src").mkdir(parents=True, exist_ok=True)
(WORK / "hk").mkdir(exist_ok=True)

for i in range(1, 8):
    urllib.request.urlretrieve(f"{REPO}/receipt{i}.jpg", WORK / "hk" / f"receipt{i}.jpg")
urllib.request.urlretrieve(f"{REPO}/ground_truth.json", WORK / "hk" / "ground_truth.json")
print("已下载:", sorted(p.name for p in (WORK / "hk").iterdir()))

## 2. 写入源码

In [ ]:
%%writefile /content/ghb/src/money.py
"""金额归一化。

票据金额的书写方式跨地区差异很大，同一个数据集内部也不统一。CORD（印尼）里
同时出现 "60.000"、"91000"、"28,000"、"Rp. 111,000"；港式票据用 "$316.10"。
若按英文习惯把 "60.000" 读成 60.00，六万会变成六十，后面所有校验都失去意义。

规则：
  1. 去掉货币符号与非数字尾巴
  2. 只剩一种分隔符且最后一段是 3 位 -> 千位分隔符，全部删去
  3. 同时出现 "." 和 ","        -> 最后出现的那个是小数点
  4. 只剩一种分隔符且最后一段不是 3 位 -> 小数点
"""
import re
from decimal import Decimal, InvalidOperation

# 允许 "Rp. 111,000"、"HK$1,234.50"、"(5.39)"、"-$16.59"
_NUM = re.compile(r"-?\d[\d.,]*")
_CURRENCY = re.compile(r"(?i)\b(?:rp|hk|idr|usd|sgd|myr|php|thb)\b\.?\s*")


def parse_amount(value, thousands_hint=None):
    """把票据上的金额写法转成 Decimal；无法解析时返回 None。

    thousands_hint: "." 或 "," 可强制指定千位分隔符（已知地区时更稳）。
    """
    if isinstance(value, bool) or value is None:
        return None
    if isinstance(value, (int, Decimal)):
        return Decimal(str(value))
    if isinstance(value, float):
        return Decimal(str(value))
    if not isinstance(value, str):
        return None

    text = _CURRENCY.sub("", value).replace("$", "").replace("￥", "").replace("¥", "")
    text = text.strip()
    negative = text.startswith("(") and text.endswith(")")
    text = text.strip("()").strip()

    m = _NUM.search(text)
    if not m:
        return None
    raw = m.group(0)
    neg = negative or raw.startswith("-")
    raw = raw.lstrip("-").rstrip(".,")
    if not raw:
        return None

    has_dot, has_comma = "." in raw, "," in raw
    if has_dot and has_comma:
        # 两种都有，最后出现的是小数点
        dec_sep = "." if raw.rfind(".") > raw.rfind(",") else ","
        thou_sep = "," if dec_sep == "." else "."
        raw = raw.replace(thou_sep, "").replace(dec_sep, ".")
    elif has_dot or has_comma:
        sep = "." if has_dot else ","
        if thousands_hint == sep:
            raw = raw.replace(sep, "")
        else:
            tail = raw.rsplit(sep, 1)[1]
            groups = raw.split(sep)
            # 每段都是 3 位且不止一段 -> 千位分隔符（"1.234.567"）
            if len(tail) == 3 and all(len(g) == 3 for g in groups[1:]):
                raw = raw.replace(sep, "")
            else:
                raw = raw.replace(sep, ".")

    try:
        amount = Decimal(raw)
    except InvalidOperation:
        return None
    return -amount if neg else amount


def q2(value):
    """量化到两位小数，便于比较。"""
    return None if value is None else value.quantize(Decimal("0.01"))


In [ ]:
%%writefile /content/ghb/src/receipt.py
"""统一的校验层：把模型的转写折算成可判定的量，跑三条校验，再跨多次识别投票。

三条校验全部在这里执行，从不进入提示词。模型不知道自己被什么标准检查，
也就无法朝那个标准编数字。
"""
import re
from decimal import Decimal

from money import parse_amount

ZERO = Decimal("0.00")


def _amounts(raw, hint):
    """把一列金额归一化成正数 Decimal。"""
    if isinstance(raw, (int, float, str, Decimal)):
        raw = [raw]
    if not isinstance(raw, list):
        return []
    out = []
    for e in raw:
        if isinstance(e, dict):
            e = e.get("amount", e.get("value", e.get("price")))
        v = parse_amount(e, hint)
        if v is not None and v != 0:
            out.append(abs(v))
    return out


def _label_confirms(label, amount):
    """标签文字里是否重复印了这笔金额。

    港式折扣行常把金额写进标签本身（"Buy 3 Save $9.8"），一个数字印了两遍，
    标签与金额栏互为佐证。模型只被要求照抄标签，并不知道这里在比对。
    """
    if not isinstance(label, str):
        return False
    for m in re.finditer(r"\d+(?:[.,]\d+)?", label):
        v = parse_amount(m.group(0))
        if v is not None and abs(v) == amount:
            return True
    return False


def _discounts(raw, rounding, hint):
    """返回 (折扣金额列表, 标签印证条数)。"""
    if isinstance(raw, (int, float, str, Decimal)):
        raw = [raw]
    if not isinstance(raw, list):
        return [], 0
    amounts, confirmed = [], 0
    for e in raw:
        label = None
        if isinstance(e, dict):
            label = e.get("label", e.get("text", e.get("description")))
            e = e.get("amount", e.get("value", e.get("price")))
        v = parse_amount(e, hint)
        if v is None or v == 0:
            continue
        v = abs(v)
        # 舍入行被误放进折扣里时可识别：它等于 |rounding|
        if rounding != 0 and v == abs(rounding):
            continue
        amounts.append(v)
        confirmed += _label_confirms(label, v)
    return amounts, confirmed


def parse_reading(payload, hint=None):
    """把一次转写折算成校验所需的量；无法解析返回 None。"""
    if not isinstance(payload, dict):
        return None
    sub = parse_amount(payload.get("subtotal"), hint)
    total = parse_amount(payload.get("total_paid"), hint)
    if sub is None and total is None:
        return None
    rounding = parse_amount(payload.get("rounding"), hint) or ZERO
    tax = parse_amount(payload.get("tax"), hint) or ZERO
    service = parse_amount(payload.get("service"), hint) or ZERO
    discounts, confirmed = _discounts(payload.get("discounts"), rounding, hint)
    items = _amounts(payload.get("items"), hint)
    disc_total = sum(discounts, ZERO)
    items_total = sum(items, ZERO)
    if sub is None:
        sub = total - tax - service + disc_total - rounding
    if total is None:
        total = sub + tax + service - disc_total + rounding
    return {
        "items_total": items_total, "n_items": len(items),
        "discount_total": disc_total, "labels_ok": confirmed,
        "subtotal": sub, "tax": tax, "service": service,
        "rounding": rounding, "total_paid": total,
    }


# 「小计」的语义跨地区不同，这是实务里必须配置的参数，不该靠猜：
#   港式超市   SUBTOTAL 印的是折扣「后」金额  -> discount_in_subtotal=True
#   印尼 CORD  subtotal_price 是折扣「前」金额 -> discount_in_subtotal=False
# 同一条公式套两地必错一头，所以把它显式化。
HK = {"discount_in_subtotal": True}
IDR = {"discount_in_subtotal": False}


def checks(rec, tol=Decimal("0.05"), locale=HK):
    """三条校验的差额。返回 None 表示这条校验在这张票上不适用。"""
    in_sub = locale["discount_in_subtotal"]
    # 校验一：实付 = 小计 + 税 + 服务费 + 舍入（小计若为折扣前，还要减去折扣）
    expect_total = (rec["subtotal"] + rec["tax"] + rec["service"] + rec["rounding"]
                    - (ZERO if in_sub else rec["discount_total"]))
    c1 = rec["total_paid"] - expect_total
    # 校验二：商品行总额（小计若为折扣后，要减去折扣）应等于小计
    c2 = None
    if rec["n_items"]:
        expect_sub = rec["items_total"] - (rec["discount_total"] if in_sub else ZERO)
        c2 = expect_sub - rec["subtotal"]
    return {
        "c1_gap": c1, "c1_pass": abs(c1) <= tol,
        "c2_gap": c2, "c2_pass": None if c2 is None else abs(c2) <= tol,
        "labels_ok": rec["labels_ok"],
    }


def verdict(rec, tol=Decimal("0.05"), locale=HK):
    """可信 / 存疑，以及失败的是哪几条。"""
    c = checks(rec, tol, locale)
    failed = []
    if not c["c1_pass"]:
        failed.append("c1")
    if c["c2_pass"] is False:
        failed.append("c2")
    return ("存疑" if failed else "可信"), failed, c


def merge(readings, tol=Decimal("0.05"), locale=HK):
    """跨多次识别取共识。

    证据强弱：自洽的优先 -> 标签印证多的优先 -> 多数投票。
    单次识别的随机误差在这里被消掉，而排序依据全部来自代码侧。
    """
    from collections import Counter

    recs = [r for r in readings if r]
    if not recs:
        return None
    ok = [r for r in recs if checks(r, tol, locale)["c1_pass"]] or recs
    consistent = [r for r in ok if checks(r, tol, locale)["c2_pass"] is not False] or ok
    best = max(r["labels_ok"] for r in consistent)
    pool = [r for r in consistent if r["labels_ok"] == best]

    out = {}
    for f in ("subtotal", "tax", "service", "rounding", "total_paid",
              "discount_total", "items_total"):
        vals = [r[f] for r in pool]
        v, n = Counter(vals).most_common(1)[0]
        out[f] = v if n > 1 or len(pool) == 1 else pool[0][f]
    out["n_items"] = pool[0]["n_items"]
    out["labels_ok"] = best
    out["reads"] = len(recs)
    out["pool"] = len(pool)
    return out


In [ ]:
%%writefile /content/ghb/src/chain.py
"""视觉模型链路：prompt -> 模型 -> JSON。

模型只做转写。提示词里没有任何等式——告诉模型的约束，模型就有动力去满足它
而不是读准，这是早期版本踩过的坑（见 docs/项目总纲.md）。

默认接 DeepSeek 视觉模型，换别家只需改 MODEL 与 build_chain 里的一行。
"""
import base64
import mimetypes
import os
from pathlib import Path

MODEL = os.environ.get("RECEIPT_MODEL", "deepseek-v4-flash-vision-exp")
CONCURRENCY = int(os.environ.get("RECEIPT_CONCURRENCY", 5))

SYSTEM_PROMPT = """你在转写一张零售票据。照抄票面上印的内容，不要做任何计算。

按 JSON 输出这些字段：
- "items"：每一行会让账单变大的金额——商品行、包装费、押金、服务费行——一行一条，正数，按票面顺序。
- "discounts"：每一行会让账单变小的金额——折扣、促销、优惠券、"% OFF"、"MEMBER PRICE"、"SAVE"、"REDEEM" 等。
  每条写成 {{"label": 该行印的文字, "amount": 金额栏的数字取正}}。舍入行不算折扣。
  折扣常印在它所属商品的下一行，标签形如 "Buy 2 Save $6"、"MB APP UPGRADE -$10"、"5% OFF"。
  label 逐字照抄，amount 取金额栏。
- "subtotal"：SUBTOTAL / 小计 行，照原样。
- "tax"：税额行（TAX、PB1、VAT、GST 等），没有则填 0。
- "service"：服务费行（SERVICE、SVC 等），没有则填 0。
- "rounding"：舍入 / 调整行，保留正负号，没有则填 0。
- "total_paid"：实付行（TOTAL、CASH、OCTOPUS、VISA、CREDIT CARD、应付金额等），照原样。

规则：
- 只转写。不要加、减、核对或调平任何总额。
- 不要为了让账单对得上而改动任何数字。票面数字若看起来对不上，也照原样报。
- 一行一条，不合并、不编造、不遗漏。
- **数字连同千位分隔符和小数点一起照抄**。票面写 "60.000" 就写 "60.000"，
  写 "28,000" 就写 "28,000"，不要替你换算或改写格式。
- 只回一个 JSON 对象，不要别的内容：
{{"items": [], "discounts": [{{"label": "", "amount": 0}}], "subtotal": 0, "tax": 0, "service": 0, "rounding": 0, "total_paid": 0}}
"""

INSTRUCTION = "照抄这张票据的各行金额与合计，按字段定义输出 JSON。"


def image_data_url(path):
    """把本地图片编码成多模态消息可用的 data URL。"""
    path = Path(path)
    mime = mimetypes.guess_type(path.name)[0] or "image/jpeg"
    return f"data:{mime};base64,{base64.b64encode(path.read_bytes()).decode('ascii')}"


def pil_data_url(img, fmt="JPEG"):
    """CORD 的图像来自内存，不落盘也能编码。"""
    import io
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format=fmt, quality=92)
    return f"data:image/{fmt.lower()};base64,{base64.b64encode(buf.getvalue()).decode('ascii')}"


def build_chain(model=MODEL, temperature=0, timeout=180):
    from langchain_core.output_parsers import JsonOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_deepseek import ChatDeepSeek

    llm = ChatDeepSeek(model=model, temperature=temperature,
                       max_retries=3, timeout=timeout)
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", [
            {"type": "text", "text": "{instruction}"},
            {"type": "image_url", "image_url": {"url": "{image_url}"}},
        ]),
    ])
    return prompt | llm | JsonOutputParser()


def read_many(chain, urls, reads=3, concurrency=CONCURRENCY):
    """对每张票据独立识别 reads 次，全部请求打包进一次并发批处理。

    返回 {票据下标: [原始 JSON, ...]}，失败的那次不计入。
    """
    payloads = [{"instruction": INSTRUCTION, "image_url": u} for _ in range(reads) for u in urls]
    owners = [i for _ in range(reads) for i in range(len(urls))]
    out = {i: [] for i in range(len(urls))}
    try:
        raw = chain.batch(payloads, config={"max_concurrency": concurrency},
                          return_exceptions=True)
    except Exception:
        return out
    for i, item in zip(owners, raw):
        if isinstance(item, dict):
            out[i].append(item)
    return out


In [ ]:
%%writefile /content/ghb/src/tamper.py
"""图像篡改：把票面上的某个金额改掉，看核验能否发现。

两种方式，用途不同：

  render_over_box  整块重绘。按框高匹配字号，取该处背景色与墨色，把新数字画上去。
                   全自动，跨 100 张图稳定，用来跑 TPR / FPR 统计量。

  copy_glyph       字形复制。从同一张票上取同字体的数字，抽出墨迹覆盖度，
                   擦掉原字后按目标处的背景与墨色重新合成。视觉真实，
                   8 倍放大才看得出，用来做定性配图。

两种都只改图像，不碰标注——否则实验无效。
"""
import random
import re
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont

PAD = 4


# ---------------------------------------------------------------- 取色

def _light_median(patch):
    """背景色：取亮于中位数的像素的中位数，避开墨迹。"""
    gray = patch.mean(axis=2)
    light = patch[gray > np.median(gray)]
    return np.median(light, axis=0) if len(light) else np.median(patch.reshape(-1, 3), axis=0)


def _ink_color(patch):
    """墨色：最暗的 5% 像素的中位数。"""
    gray = patch.mean(axis=2)
    dark = patch[gray <= np.percentile(gray, 5)]
    return np.median(dark, axis=0) if len(dark) else np.array([60.0, 40.0, 40.0])


# ---------------------------------------------------------------- 改数字

def perturb_number(text, rng):
    """改动一个数字，保持字符串长度与分隔符不变，返回 (新串, 新值-旧值 的十进制位)。

    保持长度是为了重绘后仍能塞进原来的框。
    """
    digits = [i for i, c in enumerate(text) if c.isdigit()]
    if not digits:
        return None
    # 不改首位为 0，也不把唯一一位改成 0
    for _ in range(20):
        i = rng.choice(digits)
        old = text[i]
        new = rng.choice([d for d in "0123456789" if d != old])
        if i == digits[0] and new == "0" and len(digits) > 1:
            continue
        return text[:i] + new + text[i + 1:]
    return None


# ---------------------------------------------------------------- 方式一：整块重绘

def _font(size):
    for name in ("DejaVuSansMono-Bold.ttf", "DejaVuSansMono.ttf", "DejaVuSans-Bold.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    try:  # matplotlib 自带 DejaVu，Colab 上一定有
        import matplotlib
        p = Path(matplotlib.get_data_path()) / "fonts" / "ttf" / "DejaVuSansMono-Bold.ttf"
        return ImageFont.truetype(str(p), size)
    except Exception:
        return ImageFont.load_default()


def render_over_box(img, box, new_text):
    """把 box 区域擦掉并重绘 new_text，尽量贴合原来的字号与颜色。"""
    x0, y0, x1, y1 = [int(v) for v in box]
    arr = np.array(img.convert("RGB")).astype(float)
    h, w = arr.shape[:2]
    x0, y0 = max(0, x0), max(0, y0)
    x1, y1 = min(w, x1), min(h, y1)
    if x1 - x0 < 4 or y1 - y0 < 4:
        return img, False

    ring = arr[max(0, y0 - PAD):y1 + PAD, max(0, x0 - PAD):x1 + PAD]
    bg, ink = _light_median(ring), _ink_color(arr[y0:y1, x0:x1])

    out = img.convert("RGB").copy()
    d = ImageDraw.Draw(out)
    d.rectangle([x0 - 1, y0 - 1, x1 + 1, y1 + 1], fill=tuple(int(v) for v in bg))

    # 字号按框高收敛，再按框宽微调，保证新数字不溢出
    size = max(8, int((y1 - y0) * 1.05))
    for _ in range(12):
        f = _font(size)
        tw, th = d.textbbox((0, 0), new_text, font=f)[2:]
        if tw <= (x1 - x0) and th <= (y1 - y0) * 1.25:
            break
        size -= 1
        if size < 8:
            break
    f = _font(size)
    tw, th = d.textbbox((0, 0), new_text, font=f)[2:]
    d.text((x1 - tw, y0 + ((y1 - y0) - th) // 2), new_text,
           font=f, fill=tuple(int(v) for v in ink))
    return out, True


# ---------------------------------------------------------------- 方式二：字形复制

def copy_glyph(arr, tgt, src, rng):
    """用 src 处的字形覆盖 tgt 处的字形，原地修改 arr（float RGB）。"""
    tx0, ty0, tx1, ty1 = tgt
    sx0, sy0, sx1, sy1 = src

    ring = arr[ty0 - PAD:ty1 + PAD, tx0 - PAD:tx1 + PAD]
    bg, ink = _light_median(ring), _ink_color(arr[ty0:ty1, tx0:tx1])
    noise = float(np.std(ring.mean(axis=2))) * 0.35

    ex0, ey0, ex1, ey1 = tx0 - PAD, ty0 - PAD, tx1 + PAD, ty1 + PAD
    hh, ww = ey1 - ey0, ex1 - ex0
    patch = np.broadcast_to(bg, (hh, ww, 3)) + rng.normal(0, noise, (hh, ww, 1))
    arr[ey0:ey1, ex0:ex1] = np.clip(patch, 0, 255)

    src_patch = arr[sy0:sy1, sx0:sx1]
    gray = src_patch.mean(axis=2)
    s_bg, s_ink = _light_median(src_patch).mean(), gray.min()
    alpha = np.clip((s_bg - gray) / max(s_bg - s_ink, 1e-6), 0, 1)[..., None]

    sh, sw = alpha.shape[:2]
    cy, cx = (ty0 + ty1) // 2, (tx0 + tx1) // 2
    py0, px0 = cy - sh // 2, cx - sw // 2
    region = arr[py0:py0 + sh, px0:px0 + sw]
    arr[py0:py0 + sh, px0:px0 + sw] = np.clip(region * (1 - alpha) + ink * alpha, 0, 255)


# ---------------------------------------------------------------- CORD 专用

def quad_to_box(quad):
    xs = [quad[f"x{i}"] for i in (1, 2, 3, 4)]
    ys = [quad[f"y{i}"] for i in (1, 2, 3, 4)]
    return min(xs), min(ys), max(xs), max(ys)


def find_word(valid_line, category, text):
    """在标注里找到某个类别下、文字等于 text 的词，返回它的框。

    gt_parse 里的值可能带货币前缀（"Rp 35.000"），而 OCR 把 "Rp" 和数字
    切成了两个词，所以还要按纯数字部分再找一轮。
    """
    want = text.strip()
    digits = re.sub(r"[^\d.,]", "", want).strip(".,")
    words = [(w, line) for line in valid_line
             if line.get("category") == category for w in line.get("words", [])]

    for w, _ in words:                              # 完全相等
        if w.get("text", "").strip() == want:
            return quad_to_box(w["quad"])
    for w, _ in words:                              # 词里包含整串
        if want and want in w.get("text", ""):
            return quad_to_box(w["quad"])
    if digits:                                      # 只比数字部分
        for w, _ in words:
            if re.sub(r"[^\d.,]", "", w.get("text", "")).strip(".,") == digits:
                return quad_to_box(w["quad"])
    return None


def field_to_category(field):
    """"menu[0].price" -> "menu.price"；标注里的 category 不带下标。"""
    return re.sub(r"\[\d+\]", "", field)


def tamper_cord(sample_image, valid_line, field, old_text, seed=0):
    """按字段名在 CORD 图像上改动一个金额。

    返回 (新图, 说明)；定位不到目标词时返回 (原图, None)。
    """
    rng = random.Random(seed)
    box = find_word(valid_line, field_to_category(field), old_text)
    if box is None:
        return sample_image, None
    new_text = perturb_number(old_text, rng)
    if not new_text or new_text == old_text:
        return sample_image, None
    out, ok = render_over_box(sample_image, box, new_text)
    if not ok:
        return sample_image, None
    return out, {"field": field, "old": old_text, "new": new_text, "box": box}


In [ ]:
import sys
sys.path.insert(0, "/content/ghb/src")
for m in ("money", "receipt", "chain", "tamper"):
    sys.modules.pop(m, None)
import money, receipt, chain, tamper
print("源码已加载")

## 3. 生成篡改样本（字形复制）

从同一张票上取同字体的数字，抽出墨迹覆盖度，擦掉原字后按目标处的背景色与墨色
重新合成。字体、墨色、噪点全都一致，不是画图软件重绘。

坐标来自对 receipt2.jpg 金额栏的自动字符切分：

| 位置 | 坐标 | 内容 |
|---|---|---|
| `OCTOPUS_3` | (655, 1216, 664, 1236) | `$316.10` 的首位 3 |
| `FARM_1` | (674, 823, 681, 843) | `$91.80` 的 1 |
| `FARM_8` | (695, 823, 704, 843) | `$91.80` 的 8，作字形来源 |
| `YAKULT_8` | (674, 869, 682, 889) | `$28.90` 的 8，作字形来源 |

两个样本各自只被一条校验抓到，正好论证三条校验的分工。

In [ ]:
import numpy as np
from PIL import Image

OCTOPUS_3 = (655, 1216, 664, 1236)
FARM_1    = (674, 823, 681, 843)
FARM_8    = (695, 823, 704, 843)
YAKULT_8  = (674, 869, 682, 889)

HK = Path("/content/ghb/hk")


def make(out_name, edits, note):
    arr = np.array(Image.open(HK / "receipt2.jpg").convert("RGB")).astype(float)
    rng = np.random.default_rng(0)
    for tgt, src in edits:
        tamper.copy_glyph(arr, tgt, src, rng)
    Image.fromarray(arr.astype(np.uint8)).save(HK / out_name, quality=92, subsampling=0)
    print(f"  {out_name:26s} {note}")


print("生成篡改样本：")
# 付款行虚增：小计与舍入未动，付款行不再等于 小计+舍入
make("receipt2_T1_payment.jpg", [(OCTOPUS_3, FARM_8)],
     "T1 付款行 316.10 -> 816.10（虚增 500）")
# 商品行虚增：付款行照样闭合，只有商品行解释不了小计
make("receipt2_T2_item.jpg", [(FARM_1, YAKULT_8)],
     "T2 商品行 91.80 -> 98.80（虚增 7）")

## 4. 看一眼改得像不像

In [ ]:
from IPython.display import display

files = ["receipt2.jpg", "receipt2_T1_payment.jpg", "receipt2_T2_item.jpg"]
for box, title in [((635, 1210, 730, 1242), "付款行   原图 / T1 / T2"),
                   ((640, 818, 725, 848), "商品行   原图 / T1 / T2")]:
    S = 6
    w, h = (box[2] - box[0]) * S, (box[3] - box[1]) * S
    sheet = Image.new("RGB", (w, h * 3), "white")
    for i, f in enumerate(files):
        sheet.paste(Image.open(HK / f).crop(box).resize((w, h), Image.LANCZOS), (0, i * h))
    print(title)
    display(sheet)

print("这是 6 倍放大。正常尺寸下看不出改动——肉眼不是可靠的核验手段。")

## 5. 送检

三张一起跑：未篡改的对照组、T1、T2。用 `receipt.HK` 地区参数
（港式 SUBTOTAL 是折扣**后**金额，与 CORD 相反）。

In [ ]:
import time

READS = 5
targets = [("对照组 未篡改", "receipt2.jpg", "两条校验都通过 -> 可信"),
           ("T1 付款行 316.10->816.10", "receipt2_T1_payment.jpg",
            "只有校验一失败，差 +500.00"),
           ("T2 商品行 91.80->98.80", "receipt2_T2_item.jpg",
            "只有校验二失败，差 +7.00")]

ch = chain.build_chain()
urls = [chain.image_data_url(HK / f) for _, f, _ in targets]
t0 = time.time()
readings = chain.read_many(ch, urls, reads=READS)
print(f"{len(targets)} 张 x {READS} 次，耗时 {time.time() - t0:.0f}s")
print()

for i, (name, _, expect) in enumerate(targets):
    recs = [r for r in (receipt.parse_reading(p) for p in readings[i]) if r]
    m = receipt.merge(recs, locale=receipt.HK)
    print("=" * 70)
    print(name)
    if m is None:
        print("  全部识别失败")
        continue
    v, failed, c = receipt.verdict(m, locale=receipt.HK)
    print(f"  有效识别 {m['reads']}/{READS}   参与共识 {m['pool']} 次   "
          f"标签印证 {m['labels_ok']} 条")
    print(f"  小计={m['subtotal']}  实付={m['total_paid']}  "
          f"折扣={m['discount_total']}  商品行={m['items_total']}")
    print(f"  校验一 付款行闭合      差 {c['c1_gap']:+}   "
          f"{'通过' if c['c1_pass'] else '不通过'}")
    print(f"  校验二 商品行解释小计  差 {c['c2_gap']:+}   "
          f"{'通过' if c['c2_pass'] else '不通过'}")
    print(f"  判定：{v}" + (f"（失败项 {failed}）" if failed else ""))
    print(f"  预期：{expect}")

## 6. 未篡改的 7 张 —— 港式票据上的误报基线

CORD 上标注本身就有约两成不自洽，港式这 7 张是逐张人工核对过的，
理论上误报应为 0。跑出来若不是 0，差额就来自识别误差，而非方法缺陷。

In [ ]:
urls = [chain.image_data_url(HK / f"receipt{i}.jpg") for i in range(1, 8)]
t0 = time.time()
readings = chain.read_many(ch, urls, reads=READS)
print(f"7 张 x {READS} 次，耗时 {time.time() - t0:.0f}s")
print()

flagged = 0
for i in range(7):
    recs = [r for r in (receipt.parse_reading(p) for p in readings[i]) if r]
    m = receipt.merge(recs, locale=receipt.HK)
    if m is None:
        print(f"  receipt{i + 1}.jpg  识别失败")
        continue
    v, failed, c = receipt.verdict(m, locale=receipt.HK)
    flagged += (v == "存疑")
    print(f"  receipt{i + 1}.jpg  判定={v}  c1差={c['c1_gap']:+8}  "
          f"c2差={c['c2_gap']:+8}  标签印证={m['labels_ok']}  "
          f"有效识别={m['reads']}/{READS}")
print()
print(f"误报 {flagged}/7   FPR = {flagged / 7:.0%}")

## 7. 这些结果怎么进 PPT

- 第 4 格的两张对比图 → "篡改在视觉上不明显"那一页的配图
- 第 5 格的三行判定 → 三条校验分工的表格；T1 与 T2 各自只被一条抓到是关键
- 第 6 格的 FPR → 与 CORD 的误报基线对照，说明差额来自哪里

**局限**（这句要写进 PPT，别等评委问）：查的是票面算术是否自洽，不是像素取证。
造假者若把商品行、小计、付款行一并改成彼此吻合的数字，三条校验都会通过。
本方法提高的是伪造成本——要同时骗过三条相互独立的约束，远难于改动单处。
